# 3D implementation
After the previously defined _simple_ integral evaluation schemes, it is possible to apply them to 3D cartesian Gaussians. Considering that any 3D primitive cartesian gaussian can be separated in the components:

$$
G_{ijk} = x_A^iy_A^jz_A^k \exp (-ar^2) = G_i(x, a, A_x)G_j(y, a, A_y)G_k(z, a, A_z)
$$

Where:

$$
G_i(x, a, A_x) =  x_A^i \exp (-ax_A^2)
$$

Then all the operations such as differenciation and integration can be performed in each dimension and the result is the product. Considering the normalization:

$$
G_i^{norm}(x, a, A_x) =  \frac{1}{\sqrt{N_{A,i}}}x_A^i \exp (-ax_A^2)
$$

And the 3D normalized primitive:

$$
G_{ijk} = \frac{1}{\sqrt{N_{A,i}N_{A,j}N_{A,k}}} G_i(x, a, A_x)G_j(y, a, A_y)G_k(z, a, A_z)
$$

## Overlap integrals
The Overlap integrals between two primitive Cartesian Gaussians is:

$$
S_{ab} = S_{ij}S_{kl}S_{mn}
$$

Where each component can be calculated via Obara-Saika recursion. Normalization in 3D requires that $S_{ab} = 1$. Considering the definition of the overlap integral, the simplest way to normalize is to normalize each component. This is the approach that was taken in this case. However, it must be noted that there might be other normalization schemes.

The normalized overlap between two unnormalized primitive cartesian gaussians can be defined as:

$$
S_{ab}^{norm} = \frac{1}{\sqrt{S_{ab}}}  \frac{1}{\sqrt{S_{ab}}}  S_{ij}S_{kl}S_{mn}
$$

We will use the previously defined Obara-Saika functions

In [1]:
import numpy as np
import scipy

def obara_saika_bottom_up(Ax, Bx, a, b, i, j, rd=0, verbose=False):

    max_dim = max(i,j)

    if i == j:
        max_dim += 1

    angular_momentum_matrix = np.zeros([max_dim, max_dim])

    X_ab = (Bx-Ax)
    p = a + b
    X_pa = b/p * X_ab
    X_pb = -a/p * X_ab


    angular_momentum_matrix[0,0] =  (np.pi/p)**0.5 * np.exp(-(a * b)/p * X_ab**2)

    #fill the i

    for i in range(1, max_dim):
        angular_momentum_matrix[i,0] =  X_pa * angular_momentum_matrix[i-1,0] +  1/(2*p)*((i-1) * angular_momentum_matrix[i-2,0])

    #fill the j

    for j in range(1, max_dim):
        angular_momentum_matrix[0,j] =  X_pb * angular_momentum_matrix[0,j-1] +  1/(2*p)*((j-1) * angular_momentum_matrix[0,j-2])

    for total in range(1, max_dim):
        for i in range(total, max_dim):
            angular_momentum_matrix[i,total] =  X_pa * angular_momentum_matrix[i-1,total] +  1/(2*p)*((i-1) * angular_momentum_matrix[i-2,total] + (total) * angular_momentum_matrix[i-1,total-1])
        for j in range(total, max_dim):
            angular_momentum_matrix[total,j] =  X_pb * angular_momentum_matrix[total, j-1] +  1/(2*p)*((total) * angular_momentum_matrix[total-1,j-1] +(j-1) * angular_momentum_matrix[total,j-2])

    return angular_momentum_matrix



With it we can now define the 3D functions for overlap and to calculate normalization constants:

In [2]:
def S_1D(Ax, Bx, a, b, i, j, rd=0, verbose=False):
    return obara_saika_bottom_up(Ax, Bx, a, b, i, j, rd=0, verbose=False)[i][j]

def S_3D(R_a: np.ndarray, R_b: np.ndarray, alpha, beta, a, b, c, d, e, f, rd=0, verbose=False):
    S_ab = S_1D(R_a[0], R_b[0], alpha, beta, a, b)
    S_cd = S_1D(R_a[1], R_b[1], alpha, beta, c, d)
    S_ef = S_1D(R_a[2], R_b[2], alpha, beta, e, f)
    return S_ab * S_cd * S_ef

def N_const(alpha, a, b, c):
    S_3d = S_3D([0,0,0], [0,0,0], alpha, alpha, a, b, c, a, b, c)
    return 1.0 / np.sqrt(S_3d)

And do it in an ever more clear way using a dataclass and refactoring the functions:

In [3]:
from dataclasses import dataclass

@dataclass
class Primitive:
    R: np.ndarray # of len 3
    exp: float
    quantum_numbers: np.ndarray # of len 3
    normalization_constant: float

basis_1 = Primitive([0,0,0], 0.5, [0,0,0], 1)
basis_2 = Primitive([0,0,0], 0.5, [0,0,0], 1)

def S_3D_components(basis_1: Primitive, basis_2: Primitive, rd=0, verbose=False):
    R_a = basis_1.R
    R_b = basis_2.R

    alpha = basis_1.exp
    beta = basis_2.exp

    a, c, e = basis_1.quantum_numbers
    b, d, f = basis_2.quantum_numbers

    S_ab = S_1D(R_a[0], R_b[0], alpha, beta, a, b)
    S_cd = S_1D(R_a[1], R_b[1], alpha, beta, c, d)
    S_ef = S_1D(R_a[2], R_b[2], alpha, beta, e, f)

    return([S_ab, S_cd, S_ef])

def S_3D(basis_1: Primitive, basis_2: Primitive, rd=0, verbose=False):

    S_ab, S_cd, S_ef =  S_3D_components(basis_1, basis_2)
    return S_ab * S_cd * S_ef

def N_const(basis: Primitive):

    S_3d = S_3D(basis, basis)
    return 1.0 / np.sqrt(S_3d)

basis_1.normalization_constant = N_const(basis_1)
basis_1.normalization_constant**2 * S_3D(basis_1, basis_1)

np.float64(1.0000000000000002)

## Kinetic energy integrals
In the same way as overlap, the kinetic energy integral between two Gaussians is:

$$
T_{ab} = T_{ij} S_{kl} S_{mn} + S_{ij} T_{kl} S_{mn} + S_{ij} S_{kl} T_{mn}
$$

We use Obara-Saika for one dimension as before:

In [4]:
def kinetic_energy_integrals(Ax, Bx, a, b, ii, jj, rd=0, verbose=False):

    X_ab = (Bx-Ax)
    p = a + b
    X_pa = b/p * X_ab
    X_pb = -a/p * X_ab

    max_dim = max(ii+1, jj+1)
    kinetic_energy = np.zeros([max_dim, max_dim])

    # we ask for one extra order of angular momentum due to the overlap term S_{i+1,j} and S_{i, j+1}
    recurrence_integrals = obara_saika_bottom_up(Ax, Bx, a, b, max_dim, max_dim, rd=0, verbose=False)

    #fill the T_00 case
    kinetic_energy[0,0] = (a -2 * a **2 *(X_pa ** 2 + 1/(2*p))) * recurrence_integrals[0,0]

    if ii == jj == 0:
        return kinetic_energy[0,0]

    # edge case for first row and column
    for i in range(0, max_dim-1):
        f_term = X_pa * kinetic_energy[i,0]
        s_term =  1/(2*p)*(i * kinetic_energy[i-1,0])

        t_term_1 =  b/p * (2*a * recurrence_integrals[i+1,0])
        t_term_2 =  b/p * (- i * recurrence_integrals[i-1,0])
        t_term = t_term_1 + t_term_2

        kinetic_energy[i+1,0] = f_term + s_term + t_term

    for j in range(0, max_dim-1):

        f_term = X_pb * kinetic_energy[0, j]
        s_term =  1/(2*p)*(j * kinetic_energy[0, j-1])

        t_term_1 =  a/p * (2 * b * recurrence_integrals[0, j+1])
        t_term_2 =  a/p * (- j *recurrence_integrals[0, j-1])
        t_term = t_term_1 + t_term_2

        kinetic_energy[0,j+1] = f_term + s_term + t_term

    # Iteration over the rows and columns
    for total in range(0, max_dim-1):
        j = total
        for i in range(0, max_dim-1):
            f_term = X_pa * kinetic_energy[i,j]
            s_term =  1/(2*p)*(i * kinetic_energy[i-1,j] + j * kinetic_energy[i, j-1])

            t_term_1 =  b/p * (2 * a * recurrence_integrals[i+1,j])
            t_term_2 =  b/p * (- i * recurrence_integrals[i-1,j])
            t_term = t_term_1 + t_term_2

            kinetic_energy[i+1, j] = f_term + s_term + t_term

        i = total
        for j in range(0, max_dim-1):
            f_term = X_pb * kinetic_energy[i, j]
            s_term =  1/(2*p)*(i*kinetic_energy[i-1, j] + j * kinetic_energy[i, j-1])

            t_term_1 =  a/p * (2 * b * recurrence_integrals[i, j+1])
            t_term_2 =  a/p * (- j * recurrence_integrals[i, j-1])
            t_term = t_term_1 + t_term_2

            kinetic_energy[i,j+1] = f_term + s_term + t_term


    if ii == jj and ii == max_dim-1:
        # last case
        i = max_dim -1
        j = max_dim -1

        f_term = X_pa * kinetic_energy[i-1,j]
        s_term =  1/(2*p)*((i-1) * kinetic_energy[i-2,j] + j * kinetic_energy[i-1, j-1])

        t_term_1 =  b/p * (2 * a * recurrence_integrals[i,j])
        t_term_2 =  b/p * (- (i-1) * recurrence_integrals[i-2,j])
        t_term = t_term_1 + t_term_2

        kinetic_energy[max_dim-1, max_dim-1] =  f_term + s_term + t_term

    return kinetic_energy[ii][jj], kinetic_energy

And for 3 dimensions we use the previous formula:

In [5]:
def T_3D(basis_1: Primitive, basis_2: Primitive):

    R_a = basis_1.R
    R_b = basis_2.R

    alpha = basis_1.exp
    beta  = basis_2.exp

    a, c, e = basis_1.quantum_numbers
    b, d, f = basis_2.quantum_numbers

    S_ab, S_cd, S_ef = S_3D_components(basis_1, basis_2)

    T_ab = kinetic_energy_integrals(R_a[0], R_b[0], alpha, beta, a, b, rd=0, verbose=False)
    T_cd = kinetic_energy_integrals(R_a[1], R_b[1], alpha, beta, c, d, rd=0, verbose=False)
    T_ef = kinetic_energy_integrals(R_a[2], R_b[2], alpha, beta, e, f, rd=0, verbose=False)

    return ((T_ab * S_cd * S_ef) + (S_ab * T_cd * S_ef) +  (S_ab * S_cd * T_ef ))

T_3D(basis_1, basis_2)

np.float64(4.17624599762378)

# Contracted Basis sets
It is now time to test all the previous and see if the implementation was correct. To do so we have an example of a STO-3G calculation, in which for the $H$ atom presents the following:

$$
\alpha_i = [3.42525091, 0.62391373, 0.16885540]
$$
$$
d_i=      [0.15432897, 0.53532814, 0.44463454]
$$

A contracted basis is defined as the linear combination:

$$
\mathrm{STO_{3G}}(x) = \sum_i^3 c_i · e ^{-\alpha_i(r-R_a)^2}
$$

Where $c_i$ is the product of the contraction coefficients and the normalization constant (in 3D):
$$
c_i = d_i · N_\mu
$$

In [6]:
# STO-3G data for H 1s
alphas = [3.42525091, 0.62391373, 0.16885540]
d =      [0.15432897, 0.53532814, 0.44463454]


basis_1 = Primitive(np.array([0., 0., 0.]), 3.42525091, [0,0,0], 1)
basis_2 = Primitive(np.array([0., 0., 0.]), 0.62391373, [0,0,0], 1)
basis_3 = Primitive(np.array([0., 0., 0.]), 0.16885540, [0,0,0], 1)
basis_4 = Primitive(np.array([1.4, 0., 0.]), 3.42525091, [0,0,0], 1)
basis_5 = Primitive(np.array([1.4, 0., 0.]), 0.62391373, [0,0,0], 1)
basis_6 = Primitive(np.array([1.4, 0., 0.]), 0.16885540, [0,0,0], 1)

sto_3g_h1 = [basis_1, basis_2, basis_3]
sto_3g_h2 = [basis_4, basis_5, basis_6]

normalization_constants = np.array([N_const(basis) for basis in sto_3g_h1])

# Expansion coefficients
c_mu = [d[i] * normalization_constants[i] for i in range(3)]
c_nu = [d[i] * normalization_constants[i] for i in range(3)]

Where we can build all uncontracted matrix elements with just the primitives:

In [7]:
def build_prim_matrices(basis_list_1, basis_list_2):

    dim_1 = len(basis_list_1)
    dim_2 = len(basis_list_2)

    S_prim_mat = np.zeros([dim_1, dim_2])
    T_prim_mat = np.zeros([dim_1, dim_2])

    for i in range(dim_1):
        for j in range(dim_2):
            prim_i = basis_list_1[i]
            prim_j = basis_list_2[j]

            S_ab, S_cd, S_ef = S_3D_components(prim_i, prim_j)

            # get the 3D component
            S_prim_mat[i][j] = S_ab * S_cd * S_ef
            T_prim_mat[i][j] = T_3D(prim_i, prim_j)
    return S_prim_mat, T_prim_mat


# Primitive matrices
S_prim_mat, T_prim_mat = build_prim_matrices(sto_3g_h1, sto_3g_h2)

print(S_prim_mat)

[[1.08224356e-02 2.42897324e-01 5.96154110e-01]
 [2.42897324e-01 2.16745434e+00 6.07975867e+00]
 [5.96154110e-01 6.07975867e+00 2.40459022e+01]]


In the case of a property, since there are two contracted basis, the matrix elements of the contracted basis in terms of the uncontracted matrix elements would be:

$$
S_{\mu\nu} = \sum_p^{p_{max}} \sum_q^{q_{max}} c^*_{p \mu} c_{q \nu} S_{pq}
$$

Where $c^*_{p \mu}$ refers to the contraction coefficients of basis $\mu$ and $c^*_{q \nu}$ to the ones of basis $\nu$ (where summation might not have the same dimension limit in both cases). Now we will try to get the matrix elements for two STO-3G basis separated 1.4 a.u:

$$
\mathbf{S}_{\mu\nu} =
    \begin{pmatrix}
        1.0    & 0.6593\\
        0.6593 & 1.0
    \end{pmatrix}
$$
$$
\mathbf{T}_{\mu\nu} =
    \begin{pmatrix}
        0.7600    & 0.2365\\
        0.2365 & 0.7600
    \end{pmatrix}
$$

In [8]:
def contracted_matrix_element(uncontracted_matrix, c_mu, c_nu):
# Now looping over the size of the matrices:
    contr_prop = 0.0

    max_p = len(c_mu)
    max_q = len(c_nu)

    for p in range(max_p):
        for q in range(max_q):
            contr_prop += c_nu[p] * c_mu[q] * uncontracted_matrix[p][q]


    return contr_prop

In [9]:
# so lets try to generalize to generate the contracted matrix
def S_contracted(n_primitives, primitives, contraction_coefficients):
    size = len(n_primitives)
    contracted_matrix = np.zeros([size, size])

    for mu in range(size):
        for nu in range(size):
            S_prim_mat, T_prim_mat = build_prim_matrices(primitives[mu], primitives[nu])
            contracted_matrix[mu][nu] = contracted_matrix_element(S_prim_mat, contraction_coefficients[mu], contraction_coefficients[nu])

    return contracted_matrix

def T_contracted(n_primitives, primitives, contraction_coefficients):
    size = len(n_primitives)
    contracted_matrix = np.zeros([size, size])

    for mu in range(size):
        for nu in range(size):
            S_prim_mat, T_prim_mat = build_prim_matrices(primitives[mu], primitives[nu])
            contracted_matrix[mu][nu] = contracted_matrix_element(T_prim_mat, contraction_coefficients[mu], contraction_coefficients[nu])

    return contracted_matrix

In [10]:
S_sto_3g = S_contracted([3,3], [sto_3g_h1, sto_3g_h2], [c_mu, c_nu])
S_sto_3g

array([[0.99999999, 0.6593182 ],
       [0.6593182 , 0.99999999]])

In [11]:
T_sto_3g = T_contracted([3,3], [sto_3g_h1, sto_3g_h2], [c_mu, c_nu])
T_sto_3g

array([[0.76003188, 0.23645465],
       [0.23645465, 0.76003188]])

# 3D implementation of coulomb potential and ERIS

In the following cell we are going to copy all we need to use the McMurchie-Davidson scheme implemented previously and then try to replicate the potential matrices of the same $H_2$ case.


In [12]:
# Special function-related

def pochhammer(a, k):
    return scipy.special.gamma(a+k) / scipy.special.gamma(a)

def M(a, b, x, k):
    m = 0
    for i in range(0, k):
        a_k = pochhammer(a, i)
        b_k = pochhammer(b, i)
        k_factorial = scipy.special.gamma(i+1)
        # print(f"series {i}: {a_k} {b_k} {k_factorial},  {a_k / (b_k * k_factorial)} ")
        m += a_k / (b_k * k_factorial) * x**i

    return m

def boys_hypergeom(n, x, k):
    a = n+0.5
    b = n+1.5
    return M(a, b, -x, k+10) / (2*n+1)

In [13]:
# Hermite integral-related

def R_tuv_n(p, R_pc, t_max, u_max, v_max, k_hyper):

    R_2= R_pc[0]**2 + R_pc[1]**2 +R_pc[2]**2
    X_pc, Y_pc, Z_pc = R_pc

    # the dimensions of this object are [t_max+1, u_max+1 v_max+1, n_max+1]
    # where n_max = t+u+v
    # However, when later performing the summation i must do it up to n_max, not n_max+1
    # The direct summation works here because it is initialized in np-zeros, however, it is better to consider that later in the summation function

    n_max =  t_max + u_max + v_max + 1

    R_tuv_n_array = np.zeros([t_max+1, u_max+1, v_max+1, n_max])

    for n in range(0, n_max-1):
        R_tuv_n_array[0,0,0,n] = (-2*p)**n * boys_hypergeom(n, p * R_2, k_hyper)



    # now lets get into recursion
    for t in range(0, t_max):
        for n in range(n_max-1):
            component_1 = t * R_tuv_n_array[t-1,0,0,n+1] if t >= 1 else 0
            component_2 = X_pc * R_tuv_n_array[t,0,0,n+1]
            R_tuv_n_array[t+1,0,0,n] = component_1 + component_2


    for u in range(u_max):
        for t in range(0, t_max+1):
            for n in range(n_max-1):
                component_1 = u * R_tuv_n_array[t,u-1,0,n+1] if u >= 1 else 0
                component_2 = Y_pc * R_tuv_n_array[t,u,0,n+1]
                R_tuv_n_array[t,u+1,0,n] = component_1 + component_2

    # return R_tuv_n_array

    for v in range(v_max):
        for u in range(u_max+1):
            for t in range(0, t_max+1):
                for n in range(n_max-1):
                    component_1 = v * R_tuv_n_array[t,u,v-1,n+1] if v >= 1 else 0
                    component_2 = Z_pc * R_tuv_n_array[t,u,v,n+1]
                    R_tuv_n_array[t,u,v+1,n] = component_1 + component_2
    return R_tuv_n_array

In [14]:
# Hermite to cartesian -related

def E(basis_1: Primitive, basis_2: Primitive, dim, t):
    # Calculates the Hermite coefficients of the expansion

    Ax = basis_1.R[dim]
    Bx = basis_2.R[dim]
    a = basis_1.exp
    b = basis_2.exp

    i = basis_1.quantum_numbers[dim]
    j = basis_2.quantum_numbers[dim]

    max_dim = max(i+1,j+1)

    X_ab = (Bx-Ax)
    p = a + b
    mu = (a*b)/p
    X_pa = b/p * X_ab
    X_pb = -a/p * X_ab

    #edge cases:
    if i < 0 or j < 0 or t < 0 or t > (i + j):
        return 0

    elif i==0 and j == 0 and t == 0:
        return np.exp(-mu*X_ab**2)

    # recursions
    if t > 0:
        return (i * E(i-1, j, t-1, Ax, Bx, a, b) + j * E(i, j-1, t-1, Ax, Bx, a, b))/(2*p*t)

    elif t == 0 and i > 0:
        return X_pa*E(i-1, j, t, Ax, Bx, a, b) + E(i-1, j, 1, Ax, Bx, a, b)
    elif t == 0 and j > 0:
        return X_pb*E(i, j-1, t, Ax, Bx, a, b) + E(i, j-1, 1, Ax, Bx, a, b)

def E_ab(basis_1:Primitive, basis_2: Primitive, t, u, v):

    return E(basis_1, basis_2, 0, t) * E(basis_1, basis_2, 1, u) * E(basis_1, basis_2, 2, v)

In [15]:
# Coulomb-integral related
def h_ab_Z(basis_1: Primitive, basis_2: Primitive, n_atoms:int, charge_atom:int, coord_atom:np.ndarray, k_hyper=80):

    a = basis_1.exp
    b = basis_2.exp

    r_A = basis_1.R
    r_B = basis_2.R

    i, k, m = basis_1.quantum_numbers
    j, l, n = basis_2.quantum_numbers

    t_max = i + j + 1
    u_max = k + l + 1
    v_max = m + n + 1

    p = a+b
    r_P = (a * r_A + b * r_B)/p

    h_ab_total = 0

    r_PC = r_P - coord_atom
    R_tuv_n_array = R_tuv_n(p, r_PC, t_max, u_max, v_max, k_hyper)
    charge = charge_atom

    for t in range(t_max):
        for u in range(u_max):
            for v in range(v_max):

                coefficient = E_ab(basis_1, basis_2, t, u, v)
                hermite_integral = R_tuv_n_array[t, u, v, 0]

                # print(f"{t}, {u}, {v}, {K}: {coefficient} {charge} {hermite_integral}")
                # print(f"{coefficient} ")l

                h_ab_total += coefficient * charge * hermite_integral

    return (-1)**(t_max+u_max+v_max)*2 * np.pi / p * h_ab_total

def g_abcd(basis_1, basis_2, basis_3, basis_4, k_hyper=80):

    a = basis_1.exp
    b = basis_2.exp
    c = basis_3.exp
    d = basis_4.exp

    r_A = basis_1.R
    r_B = basis_2.R
    r_C = basis_3.R
    r_D = basis_4.R

    i, k, m = basis_1.quantum_numbers
    j, l, n = basis_2.quantum_numbers
    ii, kk, mm = basis_3.quantum_numbers
    jj, ll, nn = basis_4.quantum_numbers

    t_max = i + j + 1
    u_max = k + l + 1
    v_max = m + n + 1

    tau_max = ii + jj + 1
    nu_max = kk + ll + 1
    phi_max = mm + nn + 1


    p = a+b
    r_P = (a * r_A + b * r_B)/p

    q = c+d
    r_Q = (c * r_C + d * r_D)/q

    r_PQ = r_P - r_Q

    alpha = p*q/(p+q)

    Hermite_integral = R_tuv_n(alpha, r_PQ, t_max + tau_max, u_max + nu_max, v_max + phi_max, k_hyper)

    g_abcd = 0

    for t in range(t_max):
        for u in range(u_max):
            for v in range(v_max):
                for tau in range(tau_max):
                    for nu in range(nu_max):
                        for phi in range(phi_max):
                            coefficient_1 = E_ab(basis_1, basis_2, t, u, v)
                            coefficient_2 = E_ab(basis_3, basis_4, tau, nu, phi)
                            integral = Hermite_integral[t + tau, u + nu, v + phi, 0]
                            g_abcd += coefficient_1 * coefficient_2 * integral

    return 2*np.power(np.pi,2.5)/(p*q*np.sqrt(p+q))* g_abcd


Now we are going to follow the previous procedure to try to get the nuclear atraction part of the core hamiltonian.

$$
\mathbf{V}_{\mu\nu}^{H_1} =
    \begin{pmatrix}
        -1.2266 & -0.5974\\
        -0.5974 & -0.6538
    \end{pmatrix}
$$

$$
\mathbf{V}_{\mu\nu}^{H_2} =
    \begin{pmatrix}
        -0.6538 & -0.5974\\
        -0.5974 & -1.2266
    \end{pmatrix}
$$

Recall that the potential part of the core Hamiltonian of two primitives is calculated as:

$$
h_{ab}^{NA} = -\sum_K Z_K V_{ab}^{000} = - \frac{2\pi}{p}\sum_{tuv}E_{tuv}^{ab} \sum_K Z_K R_{t,u,v}(p, \mathbf{R}_{PC_K})
$$

We will start by calculating the uncontracted $\mu\mu$ potential energy matrix due to the first nuclei (located at the same position as $\mu$):


In [16]:
def V_uncontr_segment(basis_list_1, basis_list_2, Z_atom, pos_atom):

    dim_1 = len(basis_list_1)
    dim_2 = len(basis_list_2)

    V_prim_mat = np.zeros([dim_1, dim_2])

    for i in range(dim_1):
        for j in range(dim_2):
                prim_i = basis_list_1[i]
                prim_j = basis_list_2[j]

                V_ab = h_ab_Z(prim_i, prim_j, 1, Z_atom, pos_atom)

                V_prim_mat[i][j] += V_ab

    return V_prim_mat

In [17]:
# we start by checking the analytical solution of a general 1s

basis_1 = Primitive(np.array([0, 0, 0]), 0.5, np.array([0, 0, 0]), 1)
V_uncontr_segment([basis_1], [basis_1], 1, np.array([0., 0., 0.])) # should be -1*np.pi/a = -6.283185307179586

array([[-6.28318531]])

In [18]:
# now for the three primitives:
pot = V_uncontr_segment(sto_3g_h1, sto_3g_h1, 1, np.array([0, 0., 0.]))
pot

array([[ -0.91718614,  -1.55172384,  -1.74819128],
       [ -1.55172384,  -5.03529976,  -7.92561803],
       [ -1.74819128,  -7.92561803, -18.60522467]])

In [19]:
def V_contracted(contracted_list, contraction_coefficient_list, charge_atom_list, coord_atoms_list:np.ndarray):

    size = len(contracted_list)
    n_atoms = len(charge_atom_list)

    contracted_matrix = np.zeros([size, size])



    for i in range(size):
        for j in range(size):
            for K in range(n_atoms):
                contracted_i = contracted_list[i]
                contracted_j = contracted_list[j]

                atom_charge = charge_atom_list[K]
                atom_coord = coord_atoms_list[K]

                uncontracted_segment = V_uncontr_segment(contracted_i, contracted_j, atom_charge, atom_coord)

                contracted_matrix[i][j] += contracted_matrix_element(uncontracted_segment, contraction_coefficient_list[i], contraction_coefficient_list[j])

    return contracted_matrix


In [20]:
V_1_sto3g = V_contracted([sto_3g_h1, sto_3g_h2], [c_mu, c_nu], [1], [np.array([0., 0., 0])])
V_1_sto3g

array([[-1.22661372, -0.5974173 ],
       [-0.5974173 , -0.65382715]])

In [21]:
V_2_sto3g = V_contracted([sto_3g_h1, sto_3g_h2], [c_mu, c_nu], [1], [np.array([1.4, 0., 0.])])
V_2_sto3g

array([[-0.65382715, -0.5974173 ],
       [-0.5974173 , -1.22661372]])

In [22]:
H_core_sto3g = T_sto_3g + V_1_sto3g + V_2_sto3g
H_core_sto3g

array([[-1.120409  , -0.95837996],
       [-0.95837996, -1.120409  ]])

Which matches the definition of the STO-3G core Hamiltionian for two hydrogen atoms separated 1.4 a.u.

# Contracted two electron integrals

We are going to apply the same philosopy for now in a naive manner: Build the uncontracted tensor and then contract.

$$
g_{\mu\nu\lambda\sigma} = \sum_p^{p_{max}} \sum_q^{q_{max}}\sum_r^{r_{max}} \sum_s^{s_{max}} c^*_{p \mu} c_{q \nu} c^*_{r \lambda} c_{s \sigma} (pq|rs)
$$

Should be it right?

In [23]:
def g_contracted(basis_list_1, basis_list_2, basis_list_3, basis_list_4, cont_coeff_list):
    p_max = len(basis_list_1)
    q_max = len(basis_list_2)
    r_max = len(basis_list_3)
    s_max = len(basis_list_4)

    g_final = 0

    for p in range(p_max):
        for q in range(q_max):
            for r in range(r_max):
                for s in range(s_max):
                    c_pmu = cont_coeff_list[0][p]
                    c_qnu = cont_coeff_list[1][q]
                    c_rlambda = cont_coeff_list[2][r]
                    c_ssigma = cont_coeff_list[3][s]

                    prim_i = basis_list_1[p]
                    prim_j = basis_list_2[q]
                    prim_k = basis_list_3[r]
                    prim_l = basis_list_4[s]

                    g_final += c_pmu * c_qnu * c_rlambda * c_ssigma * g_abcd(prim_i, prim_j, prim_k, prim_l)
    return g_final

And in the book it is possible to see that there are 4 unique two electron integrals due to symmetry:

$$
(11|11) = 0.7746 \,a.u
$$
$$
(11|22) = 0.5697 \,a.u
$$
$$
(21|11) = 0.4441 \,a.u
$$
$$
(21|21) = 0.2970 \,a.u
$$

In [24]:
print(f'(11|11) = {g_contracted(sto_3g_h1, sto_3g_h1, sto_3g_h1, sto_3g_h1, [c_mu, c_mu, c_mu, c_mu]):.4f}')

(11|11) = 0.7746


In [25]:
print(f'(11|22) = {g_contracted(sto_3g_h1, sto_3g_h1, sto_3g_h2, sto_3g_h2, [c_mu, c_mu, c_nu, c_nu]):.4f}')

(11|22) = 0.5697


In [26]:
print(f'(21|11) = {g_contracted(sto_3g_h2, sto_3g_h1, sto_3g_h1, sto_3g_h1, [c_nu, c_mu, c_mu, c_mu]):.4f}')

(21|11) = 0.4441


In [27]:
print(f'(21|21) = {g_contracted(sto_3g_h2, sto_3g_h1, sto_3g_h2, sto_3g_h1, [c_nu, c_mu, c_nu, c_mu]):.4f}')

(21|21) = 0.2970
